# ASKQE Evaluation with Backtranslation and Cross-Lingual QA

This notebook implements and extends the **ASKQE** (*Question Answering as Automatic Evaluation for Machine Translation*) framework.
The goal is to determine whether a monolingual English user can assess the accuracy of a machine translation (MT) by using a Question Answering (QA) system based on large language models (LLMs).

In this phase of the project, we use  **Mistral-7B-Instruct-v0.3** to compare the quality of the answers obtained through two different approaches.

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece sentence-transformers tqdm rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 48.7 MB/s eta 0:00:00


In [3]:
# all import here
from google.colab import drive
import json
import os
import torch
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# File path
BASE_PATH = "/content/drive/MyDrive/askqe project official"
PATH_EN_BASELINE = os.path.join(BASE_PATH, "QA/mistral-7b/en/Off-en-vanilla.jsonl")
PATH_ES_CLEAN = os.path.join(BASE_PATH, "QG/llama-8b/multilingual_clean_llama-8b.jsonl")

def load_jsonl(path):
    data = {}
    if not os.path.exists(path):
        print(f"File not found: {path}")
        return data
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                data[obj["id"]] = obj
    return data

# load data
baseline_en = load_jsonl(PATH_EN_BASELINE)
dataset_es_clean = load_jsonl(PATH_ES_CLEAN)

# Find common IDs
common_ids = set(baseline_en.keys()) & set(dataset_es_clean.keys())

print(f"\n✅ Dati EN caricati: {len(baseline_en)}")
print(f"✅ Dati ES (Clean) caricati: {len(dataset_es_clean)}")
print(f"✅ ID in comune: {len(common_ids)}")


✅ Dati EN caricati: 971
✅ Dati ES (Clean) caricati: 971
✅ ID in comune: 971


---

In [ ]:
# GLOBAL VARIABLES
# load and store model
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

# to compress model in 4bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print("⏳ Caricamento di Mistral-7B-v0.3 in 4-bit...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Fix for the il padding token
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

print("✅ Modello caricato con successo!")

⏳ Caricamento di Mistral-7B-v0.3 in 4-bit...


✅ Modello caricato con successo!


In [ ]:
# let's add the root folder to the system to allow importing.
if BASE_PATH not in sys.path:
    sys.path.append(BASE_PATH)

# importation of prompt!
try:
    from QA.code.prompt import qa_prompt
    print("✅ Successo: Il file 'prompt.py' è stato importato correttamente!")
    print("\n--- Anteprima del Prompt Caricato ---")
    print(qa_prompt[:200] + "...")
except ImportError as e:
    print(f"❌ Errore di importazione: {e}")
    print("Controlla che la struttura delle cartelle sia: " + os.path.join(BASE_PATH, "QA/code/prompt.py"))

✅ Successo: Il file 'prompt.py' è stato importato correttamente!

--- Anteprima del Prompt Caricato ---
Task: You will be given an English sentence and a list of relevant questions. Your goal is to generate a list of answers to the questions based on the sentence. Output only the list of answers in Pyth...


In [ ]:
# Uses Mistral-7B-Instruct to answer a set of questions given an input sentence.
# The function builds a structured QA prompt by injecting the sentence and the
# associated questions, performs deterministic inference (no sampling) for
# reproducibility, and returns only the generated answer text produced by the model.

def ask_mistral(sentence, questions):
    questions_str = str(questions)

    prompt = (
        qa_prompt
        .replace("{{sentence}}", sentence)
        .replace("{{questions}}", questions_str)
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
        )

    # Decoding
    generated = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    return generated

def calculate_similarity(baseline_ans, bt_ans):
    emb1 = sbert_model.encode(str(baseline_ans), convert_to_tensor=True)
    emb2 = sbert_model.encode(str(bt_ans), convert_to_tensor=True)
    cosine_scores = util.cos_sim(emb1, emb2) # cosine!
    return cosine_scores.item()

# load SBERT model
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ SBERT caricato e pronto per il confronto!")

### Mistral QA on Clean Backtranslated Data

This cell performs batch question answering using Mistral-7B on the clean backtranslated dataset.  
For each verified example, the model answers the original set of questions using the backtranslated English text as input.  
The generated answers are saved in JSONL format for subsequent evaluation and comparison.

This step corresponds to the QA generation stage of the ASKQE pipeline.

In [ ]:
FINAL_OUTPUT_PATH = os.path.join(BASE_PATH, "QA/mistral-7b/en_bt/answers_clean_bt.jsonl")

print(f"🚀 Avvio generazione Mistral QA (Stile Originale)dice che le domande sono su {len(verified_data)} esempi...")

with open(FINAL_OUTPUT_PATH, "w", encoding="utf-8") as f_out:
    for item in tqdm(verified_data, desc="Mistral QA", unit="ex"):
        sentence = item.get('text_en_bt')
        questions = item.get('questions')
        ex_id = item.get('id')

        if not sentence or not questions:
            continue

        try:
            # generation answers
            generated_answers = ask_mistral(sentence, questions)

            output_data = {
                "id": ex_id,
                "text_en_bt": sentence,
                "questions": questions,
                "answers": generated_answers
            }

            f_out.write(json.dumps(output_data, ensure_ascii=False) + "\n")

        except Exception as e:
            print(f"❌ Errore su ID {ex_id}: {e}")

print(f"\n✅ QA completato! Risultati salvati in: {FINAL_OUTPUT_PATH}")

### ASKQE Evaluation: Baseline vs. Clean Backtranslation

This step evaluates whether backtranslation alone introduces semantic drift in the QA outputs.  
By comparing Mistral-generated answers obtained from the original English source and from its clean backtranslated version, we assess the robustness of the ASKQE pipeline to translation-induced noise.


In [ ]:
baseline_path = os.path.join(BASE_PATH, "QA/mistral-7b/en/Off-en.jsonl")
bt_answers_path = os.path.join(BASE_PATH, "QA/mistral-7b/en_bt/answers_clean_bt.jsonl")

# seeking the Baseline into a dictionary
baseline_data = {}
with open(baseline_path, "r", encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        baseline_data[d['id']] = d['answers']

scores = []

print("📊 Analisi della similarità in corso...")

with open(bt_answers_path, "r", encoding="utf-8") as f_bt:
    for line in f_bt:
        d_bt = json.loads(line)
        ex_id = d_bt['id']

        if ex_id in baseline_data:
            ans_original = baseline_data[ex_id]
            ans_bt = d_bt['answers']
            score = calculate_similarity(ans_original, ans_bt)
            scores.append(score)

# final results
mean_similarity = np.mean(scores)
print(f"\n🎯 RISULTATO FINALE BACKTRANSLATION CLEAN")
print(f"Punteggio di Similarità Medio (SBERT): {mean_similarity:.4f}")
print(f"Esempi confrontati: {len(scores)}")

📊 Analisi della similarità in corso...

🎯 RISULTATO FINALE BACKTRANSLATION CLEAN
Punteggio di Similarità Medio (SBERT): 0.9493
Esempi confrontati: 971


In [ ]:
baseline_path = os.path.join(BASE_PATH, "QA/mistral-7b/en/Off-en-vanilla.jsonl")
bt_answers_path = os.path.join(BASE_PATH, "QA/mistral-7b/en_bt/answers_clean_bt.jsonl")

# seeking the Baseline into a dictionary
baseline_data = {}
with open(baseline_path, "r", encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        baseline_data[d['id']] = d['answers']

def calculate_metrics(original, predicted):

    original = (original)
    predicted = (predicted)

    # Exact Match
    em = 1 if original == predicted else 0

    # F1
    orig_tokens = original.split()
    pred_tokens = predicted.split()

    common = Counter(orig_tokens) & Counter(pred_tokens)
    num_same = sum(common.values())

    if len(orig_tokens) == 0 or len(pred_tokens) == 0:
        f1 = int(orig_tokens == pred_tokens)
    elif num_same == 0:
        f1 = 0
    else:
        precision = num_same / len(pred_tokens)
        recall = num_same / len(orig_tokens)
        f1 = 2 * precision * recall / (precision + recall)

    return em, f1

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

scores = []
em_scores = []
f1_scores = []
rouge_scores = []

print("📊 Analisi della similarità in corso...")

with open(bt_answers_path, "r", encoding="utf-8") as f_bt:
    for line in f_bt:
        d_bt = json.loads(line)
        ex_id = d_bt['id']

        if ex_id in baseline_data:

            ans_original = baseline_data[ex_id]
            ans_bt = d_bt['answers']

            # Se non sono liste, trasformale
            if isinstance(ans_original, str):
                ans_original = [ans_original]
            if isinstance(ans_bt, str):
                ans_bt = [ans_bt]

            for ref, pred in zip(ans_original, ans_bt):

                ref = str(ref)
                pred = str(pred)

                # SBERT
                score = calculate_similarity(ref, pred)
                scores.append(score)

                # EM + F1
                em, f1 = calculate_metrics(ref, pred)
                em_scores.append(em)
                f1_scores.append(f1)

                # ROUGE-L
                rl_score = scorer.score(ref, pred)['rougeL'].fmeasure
                rouge_scores.append(rl_score)

mean_similarity = np.mean(scores)

print(f"\n🎯 RISULTATO FINALE BACKTRANSLATION CLEAN")
print("-" * 50)
print(f"📊 SBERT Similarity:  {mean_similarity:.4f}")
print(f"✅ Exact Match (EM):  {np.mean(em_scores):.4f}")
print(f"🎯 F1-Score:          {np.mean(f1_scores):.4f}")
print(f"📝 ROUGE-L:           {np.mean(rouge_scores):.4f}")
print("-" * 50)
print(f"Totale answer pairs: {len(scores)}")

📊 Analisi della similarità in corso...

🎯 RISULTATO FINALE BACKTRANSLATION CLEAN
--------------------------------------------------
📊 SBERT Similarity:  0.9493
✅ Exact Match (EM):  0.2451
🎯 F1-Score:          0.7671
📝 ROUGE-L:           0.8212
--------------------------------------------------
Totale answer pairs: 971


### Direct Cross-Lingual QA (ES Context / EN Questions)

This cell evaluates direct cross-lingual question answering using Mistral-7B.  
The model is asked to answer English questions based on Spanish source texts, without applying backtranslation.  
This setting measures the model’s ability to perform QA across languages and serves as a comparison point against the backtranslation-based ASKQE pipeline.


In [ ]:
DIRECT_OUTPUT_PATH = os.path.join(BASE_PATH, "QA/mistral-7b/es/answers_clean_direct_es.jsonl")
os.makedirs(os.path.dirname(DIRECT_OUTPUT_PATH), exist_ok=True)

# File original multilingual
multilingual_file = os.path.join(BASE_PATH, "QG/llama-8b/multilingual_clean_llama-8b.jsonl")

print(f"🚀 Avvio QA DIRETTO (Contesto ES / Domande EN) su {len(common_ids)} esempi...")

with open(multilingual_file, "r", encoding="utf-8") as f_in, \
     open(DIRECT_OUTPUT_PATH, "w", encoding="utf-8") as f_out:

    for line in tqdm(f_in, desc="QA Cross-lingual ES", unit="ex"):
        data = json.loads(line)
        ex_id = data.get('id')
        if ex_id not in common_ids:
            continue

        sentence_es = data.get('es')
        questions_en = data.get('questions')

        if not sentence_es or not questions_en:
            continue

        try:
            ans_direct = ask_mistral(sentence_es, questions_en)

            output_entry = {
                "id": ex_id,
                "text_es": sentence_es,
                "questions_en": questions_en,
                "answers_es_raw": ans_direct
            }

            f_out.write(json.dumps(output_entry, ensure_ascii=False) + "\n")

        except Exception as e:
            print(f"❌ Errore su ID {ex_id}: {e}")

print(f"\n✅ QA Diretto completato! File salvato in: {DIRECT_OUTPUT_PATH}")

🚀 Avvio QA DIRETTO (Contesto ES / Domande EN) su 971 esempi...


QA Cross-lingual ES: 0ex [00:00, ?ex/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
QA Cross-lingual ES: 1ex [00:01,  1.82s/ex]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
QA Cross-lingual ES: 2ex [00:03,  1.71s/ex]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
QA Cross-lingual ES: 3ex [00:07,  2.94s/ex]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
QA Cross-lingual ES: 4ex [00:09,  2.47s/ex]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
QA Cross-lingual ES: 5ex [00:12,  2.49s/ex]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
QA Cross-lingual ES: 6ex [00:13,  2.27s/ex]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
QA Cross-lingual ES: 7ex [00:15,  2.14s/ex]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
QA Cross-lingual ES: 8ex [00:20,  2.88s/ex]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
QA Cr


✅ QA Diretto completato! File salvato in: /content/drive/MyDrive/askqe-project/askqe/QA/mistral-7b/es/answers_clean_direct_es.jsonl


### Evaluation of Direct Cross-Lingual QA (ES Context)

This cell evaluates the direct cross-lingual QA setting by comparing answers generated from Spanish contexts with the baseline English QA outputs.  
Answer similarity is computed using SBERT embeddings and cosine similarity, providing a quantitative measure of how closely the direct ES-based answers match the baseline EN responses.


In [ ]:
baseline_path = os.path.join(BASE_PATH, "QA/mistral-7b/en/Off-en.jsonl")
direct_es_path = os.path.join(BASE_PATH, "QA/mistral-7b/es/answers_clean_direct_es.jsonl")

baseline_data = {}
with open(baseline_path, "r", encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        baseline_data[d['id']] = d['answers']

direct_scores = []

print("📊 Analisi della similarità (DIRETTO ES) in corso...")

with open(direct_es_path, "r", encoding="utf-8") as f_dir:
    for line in f_dir:
        d_dir = json.loads(line)
        ex_id = d_dir['id']

        if ex_id in baseline_data:
            ans_original = baseline_data[ex_id]
            ans_direct = d_dir['answers_es_raw']
            emb1 = model_sbert.encode(str(ans_original), convert_to_tensor=True)
            emb2 = model_sbert.encode(str(ans_direct), convert_to_tensor=True)
            score = util.cos_sim(emb1, emb2).item()
            direct_scores.append(score)

mean_direct_sim = np.mean(direct_scores)
print(f"\n🎯 RISULTATO FINALE QA DIRETTO SPAGNOLO")
print(f"Punteggio di Similarità Medio (SBERT): {mean_direct_sim:.4f}")
print(f"Esempi confrontati: {len(direct_scores)}")

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📊 Analisi della similarità (DIRETTO ES) in corso...

🎯 RISULTATO FINALE QA DIRETTO SPAGNOLO
Punteggio di Similarità Medio (SBERT): 0.8409
Esempi confrontati: 971


In [ ]:
def calculate_metrics(original, backtranslated):
    # --- 1. Exact Match (EM) ---
    em = 1 if original.strip().lower() == backtranslated.strip().lower() else 0

    # --- 2. F1-Score (Word overlap) ---
    orig_tokens = original.lower().split()
    bt_tokens = backtranslated.lower().split()
    common = set(orig_tokens) & set(bt_tokens)

    if len(common) == 0:
        f1 = 0
    else:
        precision = len(common) / len(bt_tokens)
        recall = len(common) / len(orig_tokens)
        f1 = 2 * (precision * recall) / (precision + recall)

    return em, f1

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

em_scores = []
f1_scores = []
rouge_scores = []

print("🧪 Calcolo F1, Exact Match e ROUGE-L...")

with open(bt_answers_path, "r", encoding="utf-8") as f_bt:
    for line in f_bt:
        d_bt = json.loads(line)
        ex_id = d_bt['id']

        if ex_id in baseline_data:
            ans_original = str(baseline_data[ex_id])
            ans_bt = str(d_bt['answers'])

            # Calcolo EM e F1
            em, f1 = calculate_metrics(ans_original, ans_bt)
            em_scores.append(em)
            f1_scores.append(f1)

            # Calcolo ROUGE-L
            rl_score = scorer.score(ans_original, ans_bt)['rougeL'].fmeasure
            rouge_scores.append(rl_score)

print(f"\n📈 REPORT METRICHE DI CONFRONTO:")
print("-" * 40)
print(f"✅ Exact Match (EM):     {np.mean(em_scores):.4f}")
print(f"🎯 F1-Score (Overlap):  {np.mean(f1_scores):.4f}")
print(f"📝 ROUGE-L:             {np.mean(rouge_scores):.4f}")
print("-" * 40)
print(f"Totale esempi analizzati: {len(em_scores)}")

🧪 Calcolo F1, Exact Match e ROUGE-L...

📈 REPORT METRICHE DI CONFRONTO:
----------------------------------------
✅ Exact Match (EM):     0.2503
🎯 F1-Score (Overlap):  0.6473
📝 ROUGE-L:             0.8205
----------------------------------------
Totale esempi analizzati: 971
